# 01 — UltraFeedback Data Exploration

**Project**: LLM Fine-Tuning with DPO  
**Author**: Pranay Hedau  
**Runs on**: Mac (CPU only, no GPU needed)  

### What this notebook covers
1. Load UltraFeedback from HuggingFace and understand its structure
2. Analyze chosen vs rejected response characteristics
3. Understand what makes a response 'chosen' vs 'rejected'
4. Decide which 10K subset to use for DPO training
5. Save the subset to disk for Kaggle upload

### Why EDA before training?
Garbage in, garbage out. If we don't understand the preference signal in the data,
we can't trust our training results or debug failures. As a standard practise we perform EDA before training.

## 1. Imports & Setup

In [ ]:
import os
import re
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import Counter
from datasets import load_dataset
from dotenv import load_dotenv

load_dotenv('../.env')

plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

SEED = 42
np.random.seed(SEED)

print('Setup complete.')

## 2. Load UltraFeedback

UltraFeedback contains 64K examples. Each example has:
- `prompt` — the instruction given to the model
- `chosen` — the better response (list with role/content)
- `rejected` — the worse response (list with role/content)

This is exactly the format DPOTrainer expects so no conversion needed.

In [ ]:
print('Loading UltraFeedback from HuggingFace...')
print('This may take 1-2 minutes on first run (downloads ~500MB)')
print()

# HuggingFace TRL-ready version of UltraFeedback
# Already formatted as {prompt, chosen, rejected} — perfect for DPOTrainer
dataset = load_dataset(
    'trl-lib/ultrafeedback_binarized',
    split='train'
)

print(f'Total examples : {len(dataset)}')
print(f'Features       : {list(dataset.features.keys())}')
print()
print('First example keys:', list(dataset[0].keys()))

## 3. Understand the Data Structure

Before any analysis, we need to understand exactly what each field looks like.

In [ ]:
# Look at one example in detail
example = dataset[0]

print('=== PROMPT ===')
print(example['prompt'][:300])
print()
print('=== CHOSEN RESPONSE ===')
# chosen is a list of {role, content} dicts
if isinstance(example['chosen'], list):
    for turn in example['chosen']:
        print(f"[{turn['role']}]: {turn['content'][:200]}")
else:
    print(str(example['chosen'])[:300])
print()
print('=== REJECTED RESPONSE ===')
if isinstance(example['rejected'], list):
    for turn in example['rejected']:
        print(f"[{turn['role']}]: {turn['content'][:200]}")
else:
    print(str(example['rejected'])[:300])

In [ ]:
# Convert to pandas for easier analysis
def extract_text(field):
    """Extract plain text from chosen/rejected field regardless of format."""
    if isinstance(field, list):
        # Format: [{role: 'user', content: '...'}, {role: 'assistant', content: '...'}]
        for turn in field:
            if turn.get('role') == 'assistant':
                return turn.get('content', '')
        return field[-1].get('content', '') if field else ''
    return str(field)

# Build flat dataframe from first 20K for fast analysis
print('Building analysis dataframe...')
sample = dataset.select(range(min(20000, len(dataset))))

df = pd.DataFrame({
    'prompt':   [ex['prompt'] for ex in sample],
    'chosen':   [extract_text(ex['chosen']) for ex in sample],
    'rejected': [extract_text(ex['rejected']) for ex in sample],
})

# Compute length features
df['prompt_len']   = df['prompt'].str.split().str.len()
df['chosen_len']   = df['chosen'].str.split().str.len()
df['rejected_len'] = df['rejected'].str.split().str.len()
df['len_diff']     = df['chosen_len'] - df['rejected_len']

print(f'Dataframe shape: {df.shape}')
df.head(3)

## 4. Basic Dataset Statistics

In [ ]:
print('=== DATASET OVERVIEW ===')
print(f'Total examples analyzed : {len(df)}')
print(f'Null prompts            : {df["prompt"].isnull().sum()}')
print(f'Null chosen             : {df["chosen"].isnull().sum()}')
print(f'Null rejected           : {df["rejected"].isnull().sum()}')
print(f'Empty chosen responses  : {(df["chosen"].str.len() == 0).sum()}')
print(f'Empty rejected responses: {(df["rejected"].str.len() == 0).sum()}')
print()
print('=== LENGTH STATISTICS (word count) ===')
stats = df[['prompt_len', 'chosen_len', 'rejected_len']].describe().round(1)
print(stats)

## 5. Response Length Analysis

**Key question**: Are chosen responses systematically longer than rejected ones?  
This matters because we don't want our model to learn 'longer = better' that's a length bias, not a quality signal.

**What to look for**: If `len_diff` is close to 0 on average, the dataset has low length bias. Good.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Chosen vs Rejected length distributions
axes[0].hist(df['chosen_len'].clip(0, 500), bins=50,
             alpha=0.7, color='#2ecc71', label='Chosen', edgecolor='white')
axes[0].hist(df['rejected_len'].clip(0, 500), bins=50,
             alpha=0.7, color='#e74c3c', label='Rejected', edgecolor='white')
axes[0].set_title('Response Length Distribution', fontweight='bold')
axes[0].set_xlabel('Word Count (clipped at 500)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Length difference (chosen - rejected)
axes[1].hist(df['len_diff'].clip(-300, 300), bins=60,
             color='#3498db', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='black', linestyle='--', linewidth=1.2, label='No difference')
axes[1].axvline(df['len_diff'].mean(), color='red', linestyle='-',
                linewidth=1.5, label=f'Mean: {df["len_diff"].mean():.1f}')
axes[1].set_title('Length Difference\n(Chosen - Rejected)', fontweight='bold')
axes[1].set_xlabel('Word Count Difference')
axes[1].legend(fontsize=8)

# Prompt length distribution
axes[2].hist(df['prompt_len'].clip(0, 200), bins=50,
             color='#9b59b6', edgecolor='white', alpha=0.85)
axes[2].axvline(df['prompt_len'].mean(), color='black', linestyle='--',
                linewidth=1.2, label=f'Mean: {df["prompt_len"].mean():.0f}')
axes[2].set_title('Prompt Length Distribution', fontweight='bold')
axes[2].set_xlabel('Word Count')
axes[2].legend(fontsize=8)

plt.suptitle('UltraFeedback — Response Length Analysis', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
os.makedirs('../results', exist_ok=True)
plt.savefig('../results/01_length_analysis.png', bbox_inches='tight')
plt.show()

print(f'Mean length difference (chosen - rejected): {df["len_diff"].mean():.1f} words')
pct_chosen_longer = (df['len_diff'] > 0).mean() * 100
print(f'Chosen is longer than rejected            : {pct_chosen_longer:.1f}% of the time')
print()
if abs(df['len_diff'].mean()) < 30:
    print('FINDING: Low length bias. The preference signal is quality-based, not length-based. Good.')
else:
    print('FINDING: Moderate length bias detected. Worth monitoring during training.')

## 6. What Makes a Response 'Chosen'?

We will manually inspect several examples to build intuition.

In [ ]:
print('=== SIDE BY SIDE COMPARISON: 3 RANDOM EXAMPLES ===')
print()

sample_idx = [42, 137, 891]

for i, idx in enumerate(sample_idx):
    row = df.iloc[idx]
    print(f'{'='*70}')
    print(f'EXAMPLE {i+1}')
    print(f'{'='*70}')
    print(f'PROMPT ({row["prompt_len"]} words):')
    print(f'  {row["prompt"][:200]}...' if len(row['prompt']) > 200 else f'  {row["prompt"]}')
    print()
    print(f'CHOSEN ({row["chosen_len"]} words):')
    print(f'  {row["chosen"][:300]}...' if len(row['chosen']) > 300 else f'  {row["chosen"]}')
    print()
    print(f'REJECTED ({row["rejected_len"]} words):')
    print(f'  {row["rejected"][:300]}...' if len(row['rejected']) > 300 else f'  {row["rejected"]}')
    print()

## 7. Quality Signal Analysis

We look for surface patterns that correlate with chosen vs rejected:
- Do chosen responses use more structured formatting (lists, steps)?
- Are rejected responses more likely to contain hedging language?
- Do chosen responses have fewer refusals?

This tells us what behavioral patterns DPO will learn to reinforce.

In [ ]:
# Surface pattern analysis
def has_list(text):
    return bool(re.search(r'(\n\s*[-•*]|\n\s*\d+\.)', text))

def has_hedging(text):
    hedges = ['i cannot', 'i am not able', 'i don\'t know', 
              'i\'m not sure', 'unfortunately', 'i apologize']
    text_lower = text.lower()
    return any(h in text_lower for h in hedges)

def has_code_block(text):
    return '```' in text

def starts_with_sure(text):
    return text.lower().strip().startswith(('sure', 'of course', 'certainly'))

metrics = {
    'Has list/structure'  : (df['chosen'].apply(has_list).mean(),
                             df['rejected'].apply(has_list).mean()),
    'Has hedging language': (df['chosen'].apply(has_hedging).mean(),
                             df['rejected'].apply(has_hedging).mean()),
    'Has code block'      : (df['chosen'].apply(has_code_block).mean(),
                             df['rejected'].apply(has_code_block).mean()),
    'Starts with Sure/Of course': (df['chosen'].apply(starts_with_sure).mean(),
                                   df['rejected'].apply(starts_with_sure).mean()),
}

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(metrics))
width = 0.35

chosen_vals   = [v[0] * 100 for v in metrics.values()]
rejected_vals = [v[1] * 100 for v in metrics.values()]

bars1 = ax.bar(x - width/2, chosen_vals,  width, label='Chosen',   color='#2ecc71', edgecolor='white')
bars2 = ax.bar(x + width/2, rejected_vals, width, label='Rejected', color='#e74c3c', edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(list(metrics.keys()), fontsize=9)
ax.set_ylabel('Percentage of responses (%)')
ax.set_title('Surface Quality Patterns: Chosen vs Rejected', fontweight='bold', pad=12)
ax.legend()

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('../results/02_quality_patterns.png', bbox_inches='tight')
plt.show()

print('Pattern Analysis:')
for pattern, (c, r) in metrics.items():
    diff = (c - r) * 100
    direction = 'more' if diff > 0 else 'less'
    print(f'  {pattern:<30}: chosen {direction} by {abs(diff):.1f}%')

## 8. Select 10K Training Subset

I am selecting 10K out of 64K for training. Why 10K?
- Full 64K takes ~14 hours on Kaggle T4 (exceeds session limit)
- 10K takes ~3-4 hours which fits within one Kaggle session

**Subset selection strategy**: Filter for examples where the quality gap between chosen and rejected is meaningful — we exclude pairs where both responses are nearly identical in length (low signal pairs).

In [ ]:
print('Selecting 10K training subset...')

# Load full dataset indices
full_df = pd.DataFrame({
    'idx':          range(len(dataset)),
    'prompt_len':   [len(ex['prompt'].split()) for ex in dataset],
    'chosen_len':   [len(extract_text(ex['chosen']).split()) for ex in dataset],
    'rejected_len': [len(extract_text(ex['rejected']).split()) for ex in dataset],
})
full_df['len_diff'] = (full_df['chosen_len'] - full_df['rejected_len']).abs()

# Filter criteria:
# 1. Prompt not too short (< 5 words = likely noise) or too long (> 300 = may exceed token limit)
# 2. Both responses non-empty
# 3. Responses not identical length (low signal)
filtered = full_df[
    (full_df['prompt_len'] >= 5) &
    (full_df['prompt_len'] <= 300) &
    (full_df['chosen_len'] >= 10) &
    (full_df['rejected_len'] >= 10) &
    (full_df['len_diff'] >= 3)    # some measurable difference
]

print(f'Total examples            : {len(full_df)}')
print(f'After filtering           : {len(filtered)}')
print(f'Filter retention rate     : {len(filtered)/len(full_df)*100:.1f}%')

# Sample 10K with fixed seed for reproducibility
subset_indices = filtered['idx'].sample(n=10000, random_state=SEED).tolist()
train_subset = dataset.select(subset_indices)

print(f'Final training subset size: {len(train_subset)}')

In [ ]:
# Save subset to disk for Kaggle upload
os.makedirs('../data', exist_ok=True)
train_subset.to_json('../data/ultrafeedback_10k.jsonl', orient='records', lines=True)

# Also save subset indices for reproducibility
with open('../data/subset_indices.json', 'w') as f:
    json.dump(subset_indices, f)

print('Saved:')
print(f'  ../data/ultrafeedback_10k.jsonl  ({os.path.getsize("../data/ultrafeedback_10k.jsonl") / 1e6:.1f} MB)')
print(f'  ../data/subset_indices.json')
print()
print('Next step: Upload ultrafeedback_10k.jsonl to Kaggle Datasets')
print('Then use it in notebook 03_dpo_training.ipynb on Kaggle T4')

## 9. Final Subset Quality Check

In [ ]:
# Verify the saved subset
subset_df = pd.read_json('../data/ultrafeedback_10k.jsonl', lines=True)

subset_df['chosen_text']   = subset_df['chosen'].apply(extract_text)
subset_df['rejected_text'] = subset_df['rejected'].apply(extract_text)
subset_df['chosen_wc']     = subset_df['chosen_text'].str.split().str.len()
subset_df['rejected_wc']   = subset_df['rejected_text'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(subset_df['chosen_wc'].clip(0, 600), bins=50,
             alpha=0.7, color='#2ecc71', label='Chosen', edgecolor='white')
axes[0].hist(subset_df['rejected_wc'].clip(0, 600), bins=50,
             alpha=0.7, color='#e74c3c', label='Rejected', edgecolor='white')
axes[0].set_title('10K Subset — Response Lengths', fontweight='bold')
axes[0].set_xlabel('Word Count')
axes[0].legend()

prompt_wc = subset_df['prompt'].str.split().str.len()
axes[1].hist(prompt_wc.clip(0, 300), bins=50,
             color='#9b59b6', edgecolor='white', alpha=0.85)
axes[1].axvline(prompt_wc.mean(), color='black', linestyle='--',
                label=f'Mean: {prompt_wc.mean():.0f}')
axes[1].set_title('10K Subset — Prompt Lengths', fontweight='bold')
axes[1].set_xlabel('Word Count')
axes[1].legend()

plt.suptitle('Final 10K Training Subset Quality Check', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/03_subset_quality.png', bbox_inches='tight')
plt.show()

print('10K Subset Stats:')
print(f'  Chosen avg length  : {subset_df["chosen_wc"].mean():.0f} words')
print(f'  Rejected avg length: {subset_df["rejected_wc"].mean():.0f} words')
print(f'  Prompt avg length  : {prompt_wc.mean():.0f} words')
print(f'  Null values        : {subset_df.isnull().sum().sum()}')

## 10. Key Findings Summary

In [ ]:
chosen_longer_pct = (df['len_diff'] > 0).mean() * 100

print('=' * 70)
print(f'{"DATA EXPLORATION KEY FINDINGS":^70}')
print('=' * 70)

findings = [
    ('Dataset Size',
     f'{len(dataset):,} total preference pairs',
     'Using 10K filtered subset for training (~3-4hr on Kaggle T4).'),

    ('Data Format',
     'prompt / chosen / rejected — matches DPOTrainer exactly',
     'No format conversion needed. Load directly with datasets library.'),

    ('Length Bias',
     f'Chosen longer {chosen_longer_pct:.0f}% of the time, mean diff={df["len_diff"].mean():.1f} words',
     'Mild length bias. Monitor during training. Not severe enough to filter.'),

    ('Quality Signal',
     'Chosen responses more structured, fewer hedges',
     'DPO will learn: more structure + directness = preferred behavior.'),

    ('Subset Strategy',
     '10K filtered on prompt/response length + minimum quality gap',
     'Saved to data/ultrafeedback_10k.jsonl — upload to Kaggle next.'),
]

for i, (topic, finding, action) in enumerate(findings, 1):
    print(f'\n{i}. {topic}')
    print(f'   Finding : {finding}')
    print(f'   Action  : {action}')

print()
print('=' * 70)
print('Next: 02_sft_baseline.ipynb — Train SFT reference model on Kaggle T4')
print('=' * 70)